# Part 8 - The ENCODE MotifCompendium pipeline

The MotifCompendium packaged was used to create the **official ENCODE TF ChIP-seq CWM motif set**, and **ATAC-seq+DNase-seq CWM motif set**. The following tutorial outlines the pipeline process that was used to create the initial set, and how to use the command line interface (CLI) version of the pipeline (under `./pipeline`).

### Overview of ENCODE pipeline
- **Step 0**: Set basic pipeline arguments
- **Step 1**: Load TF-MoDISco motifs and build MotifCompendium object
- **Step 2**: Annotate motifs
- **Step 3**: Filter motifs
- **Step 4**: Cluster motifs (cluster, metacluster, sub-cluster)
- **Step 5**: Average clusters (cluster, metacluster, sub-cluster)
- **Step 6**: Annotate clusters, metaclusters, sub-clusters
- **Step 7**: Filter clusters, metaclusters, sub-clusters
- **Step 8**: Visualize motifs and clusters
- (Optional) **Step 9**: Further modify pipeline configurations

### How to use the ENCODE pipeline
- Command line interface (CLI): Use the CLI in `./pipeline/pipeline.py`
    - This tutorial will cover how to use the basic arguments in `pipeline.py`:
- *(Advanced)* Additional pipeline configurations: Change additional parameters and configurations in `./pipeline/configs.py`

#### Example:

In [ ]:
python ./pipeline/pipeline.py \
    -o "./tutorials/output" \
    -ih "./tutorials/data/ENCSR155NPL_counts_motifs.h5" "./tutorials/data/ENCSR279SXQ_counts_motifs.h5" \
    -nh "ENCSR155NPL" "ENCSR279SXQ" \
    -m "./tutorials/data/8_metadata.tsv"
    -r "./pipeline/data/HUMAN-JASPAR2024-HOCOMOCOv13.meme.txt"
    --sim-threshold 0.88 \
    --sim-threshold-force \
    --cluster-recursive \
    --html-motif-table \
    --html-motif-removed \
    --html-cluster-table \
    --html-cluster-removed \
    --html-motif-collection \
    -ch 1000 \
    -cp 16 \
    --use-gpu \
    -t \
    -v \

## 8-0. Set initial pipeline arguments

We set the basic set of pipeline arguments. (We will introduce more arguments as we proceed with each step)!

- **Output directory**: Path to output directory *(required)*
- **Max chunk size**: Maximum number of motifs to process at a time (higher=faster; will depend on CPU/GPU memory) *(default: 1000)*
- **Max CPUs**: Maximum number of CPUs to use (will not go beyond the max number of CPUs of the device) *(default: 1)*
- **Use GPU**: Use GPU for processing *(**strongly** recommended)* *(default: False)*
- **Verbose**: Print verbose output, for debugging *(default: False)*
- **Time**: Print time taken for each step, for debugging *(default: False)*

> CLI instructions:

```
python pipeline.py \
    -o, --output-dir {Path to output directory; str} \
    -ch, --max-chunk {Maximum number of motifs to process at a time; int} \
    -cp, --max-cpus {Maximum number of CPUs to use; int} \
    --use-gpu {Use GPU for processing; bool} \
    -t, --time {Print time take for each step; bool} \
    -v, --verbose {Print verbose output; bool} \
    ...
```

## 8-1. Load motifs and build MotifCompendium object

### 8-1-1. Loading motifs: PFM, MEME

First, we load the set of **starting motifs**. We can load two major motif file types:
- **PFM, MEME file**: Path(s) to the input PFM or MEME text file(s).
    - `name`: Motif name
    - `posneg`: Positive/Negative motif, `{pos/neg}`

> CLI instructions:
```
python pipeline.py \
    -ip, --input-pfms {Path(s) to the input PFM or MEME text file(s); List(str)} \
    ...
```

### 8-1-2. Loading motifs: TF-MoDISco (lite)
- **TF-MoDISco (lite) object (.h5)**: Path(s) to the input TF-MoDISco (lite) H5 object(s) (and file nickname; *required, as all TF-MoDISco objects have the same naming scheme: e.g., `pos_pattern_0`*)
    - `name`: Motif name, `{pos/neg}.pattern_{#}`
    - `posneg`: Positive/Negative motif, `{pos/neg}`
    - `num_seqlets`: Number of seqlets
    - `model`: Model name, `{input_name}`
    - `avg_contrib`: Mean contribution score
    - `avg_dist_from_summit`: Mean distance from peak summit 

> CLI instructions:
```
python pipeline.py \
    -ih, --input-h5s {Path(s) to the input Modisco H5 file(s); List(str)} \
    -nh, --input-names {Nickname(s) of the input Modisco H5 file(s); List(str)} \
    ...
```

#### 8-1-3. (Optional) Loading Metadata

We can load supplementary **metadata**, as a CSV or TSV file. This can include information such as: experiment ID, target, biosample, etc. We can do this in one of two ways:
- (Optional) **Metadata**: A table containing metadata of the motifs, as a CSV or TSV file. Can be formatted as follows:
    - **Per TF-MoDISco H5 object**: Each row represents metadata for **one TF-MoDISco H5 object** *(default)*
    - **Per motif**: Each row represents metadata for **one motif** - if number of rows = number of motifs

> CLI instructions:
```
python pipeline.py \
    -m, --metadata {Path to metadata file per h5 or per motif, as CSV or TSV format; str} \
    ...
```

#### 8-1-4. (Optional) Loading Pre-built MotifCompendium objects

To prevent re-building a new MotifCompendium object every time, we can also load **pre-built MotifCompendium objects**:
1. **Pre-built MotifCompendium object (.mc)**: Path to a pre-built MotifCompendium object, already containing the starting motifs
2. **Pre-built old MotifCompendium object (.mc)**: Path to a pre-built, older version MotifCompendium object, already containing the starting motifs

> CLI instructions:
```
python pipeline.py \
    -im, --input-mc {Path to the input MotifCompendium object; str} \
    ...
```

We also allow loading of **previous generations** of MotifCompendium objects:
- *Note: Currently supports old MotifCompendium object from v0.0*

> CLI instructions:
```
python pipeline.py \
    -io, --input-old-mc {Path to the input old MotifCompendium object; str} \
    ...
```

### 8-1-5. Output:

This constructs the **initial MotifCompendium object**. Output is saved at: 
- **Initial MotifCompendium**: `${OUTPUT_DIR}/motifcompendium.mc`

## 8-2. Annotate motifs

### 8-2-1. Main reference motif database

Second, we **annotate motifs** based on a **known reference motif database**. (Refer to **Tutorial #5: Motif annotation**, for more detail)

We can specify the reference motif database in one of 3 ways: 
1. **[Default]**: Uses the union of motifs from: (1) JASPAR 2024 - CORE HUMAN, (2) HOCOMOCO V13 CORE 
2. **PFM, MEME format**: Path to a text file in PFM or MEME format *(with 'pfm' or 'meme' in file name)* *(optional)*
3. **MotifCompendium object (.mc)**: Path to a MotifCompendium object with the reference motifs *(optional)*

> CLI instructions:
```
python pipeline.py \
    -r, --reference {Path to reference motifs file, in MEME, PFM, or MotifCompendium object format; str} \
    ...
```

### 8-2-2. (Optional) Additional reference motif databases

Additionally, we can annotate motifs based on **multiple reference motif databases**: by specifying additional databases.

> CLI instructions:
```
python pipeline.py \
    --add-reference {Path(s) to additional reference motifs file(s), in MEME, PFM, or MotifCompednium object format; List(str); optional} \
    ...
```

### 8-2-3. Output:

This adds the following columns to the MotifCompendium metadata and images:
- `reference_name{i}`: Best matching motif name 
- `reference_score{i}`: Best matching similarity score
- `reference_logo{i}`: Best matching motif logo image

Where `{i}` helps identify single motifs vs. composite motifs (e.g., `i=2` suggests a bi-part composite motif, composed of two known motifs; default `i=2`)

Output is saved at: 
- **Labeled MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_labeled.mc`

## 8-3. Filter motifs

Third, we **remove unwanted, low confidence motifs**. (Refer to **Tutorial #7: Motif filtering**, for more detail)

We filter motifs over three steps:
1. **Initial filter**: Flag motifs for removal
2. **Override filter**: Override based on annotation match
3. (Optional) **Strict filter**: Remove even annotated motifs

### 8-3-1. (Optional) No filter

If we already know that **all motifs are good**, and none need to be filtered, we can turn off filtering entirely. Otherwise, filtering is run *by default*.

> CLI instructions:
```
python pipeline.py \
    --no-filter {Do NOT apply any filters; bool} \
    ...
```

### 8-3-2. Initial filter flag

We flag for removal the following **types of unwanted motifs**:

1. **Single peak**: A single, sharp peak (e.g., __A_)
2. **Random, noisy chaos**: Random noise
3. **Sharp uncertain peaks**: High contribution peak, but equal across all bases
4. **Broad single nucleotide**: Broad single nucleotide repeats (e.g., AAAAAA)
5. **Broad GC/AT bias**: GC/AT noise, with consistently similar contribution between G/C, A/T
6. **Broad dinucleotide repeats**: Repeated nucleotide pair (e.g., CpG: GCGCGCGC)
7. **Few seqlets**: A motif constructed from too few seqlets
8. **Positive/Negative inverted**: Categorized as a positive/negative pattern by modisco, but sum of contributions is negative/positive
9. **Truncated motif**: Motif is not centered (pushed left/right) and truncated short by window length

> CLI instructions: None

*Initial filtering is run automatically as default, unless specified with `--no-filter` (refer to 8-3-1.)*

### 8-3-3. Override filter, based on annotation match

We would like to keep motifs that have a **strong match with known, high confidence motifs** in the reference motif database. We do this by filtering based on the maximum similarity score with the main reference database. A high match is considered the following:
- **Single motif**: Similarity score > 0.88 *(default)*
- **Composite motif**: Similarity score > 0.7 *(default)*

> CLI instructions: None

*Override filter is run automatically as default, unless specified with `--no-filter` (refer to 8-3-1.)*

### 8-3-4. (Optional) Strict filter, removing even annotated motifs

Not all motifs in an external database are perfect. We apply a strict filter to override and remove even motifs that have a good annotated match. The thresholds are more conservative than **"8-2-1. Initial filter"**:
1. **Random, noisy chaos**: Random noise
2. **Sharp uncertain peaks**: High contribution peak, but equal across all bases
3. **Broad single nucleotide**: Broad single nucleotide repeats (e.g., AAAAAA)
4. **Broad GC/AT bias**: GC/AT noise, with consistently similar contribution between G/C, A/T
5. **Broad dinucleotide repeats**: Repeated nucleotide pair (e.g., CpG: GCGCGCGC)
6. **Few seqlets**: A motif constructed from too few seqlets
7. **Positive/Negative inverted**: Categorized as a positive/negative pattern by modisco, but sum of contributions is negative/positive
8. **Truncated motif**: Motif is not centered (pushed left/right) and truncated short by window length

> CLI instructions:
```
python pipeline.py \
    --strict-filter {Apply strict filter, overriding annotation; bool} \
    ...
```

### 8-3-5. Remove flagged motifs

Any flags still flagged for removal are removed from the main MotifCompendium object, and separately stored as two distinct MotifCompendium objects:
- **Filtered** (i.e., passed): Motifs that passed by filtering *(from hereon, only filter-passed motifs are used)*
- **Removed**: Motifs removed by filtering

> CLI instructions: None

*Removal is run automatically as default, unless specified with `--no-filter` (refer to 8-3-1.)*

### 8-3-6. Output:
This adds the following columns to the MotifCompendium metadata:
- **Motif metrics**: Quantitative metrics used to filter
    - `motif_entropy`
    - `weighted_base_entropy`
    - `posbase_entropy_ratio`
    - `copair_entropy_ratio`
    - `dinuc_entropy_ratio`
    - `posneg_inverted`
    - `truncated`
- `flag_remove`: Final flag for removal (`True`=Remove; `False`=Keep)

Output is saved at: 
- **Filtered** (i.e., passed): `${OUTPUT_DIR}/motifcompendium_filtered.mc`
- **Removed**: `${OUTPUT_DIR}/motifcompendium_removed.mc`

## 8-4. Cluster motifs

Fourth, we **cluster** the motifs. (Refer to **Tutorial #3: Motif clustering**, for more detail)

We perform **up to four** clustering processes: 
1. *(Optional)* **Pre-group motifs**: Group motifs, based on prior categorization in motif metadata
2. **Motif clustering**: Cluster motifs --> clusters
3. *(Optional)* **Meta-clustering**: Cluster clusters --> meta-clusters
4. *(Optional)* **Sub-clustering**: Cluster motifs within a cluster --> sub-clusters

*[Note relationship between each cluster: Motifs ⊆ Sub-clusters ⊆ Clusters ⊆ Meta-clusters]*

Each clustering process involves **up to three** clustering steps:
1. **Initial clustering**: Apply clustering, based on similarity matrix
2. *(Optional)* **Recursive clustering**: Recursively apply the clustering, until convergence
3. *(Optional)* **Forced clustering**: Ensure all clusters have a similarity below a forced threshold, by applying a densely connected components (DCC) clustering (if recursive clustering is `True`, also applied recursively)

### 8-4-1. (Optional) Pre-group motifs

Motifs often have some **prior categorization** before any clustering, just from the data source. For example, for DNase-seq/ATAC-seq motifs, motifs can be pre-catagorized by their source cell type (e.g., neuron, heart). Or, when managing subpatterns of a TF-MoDISco motif, motifs can be pre-categorized by their parent pattern (e.g., `pos_pattern_0`). 

These categorizations are not derived from our motif similarity calculation, but already known during data import. We would still like to incorporate this information during clustering.

Generally, there are two groups of prior categorization. All prior categorization must be included as a column in the MotifCompendium metadata table `{prior_categorization_col}`:
- **Cluster within**: Slice the motif set into distinct subsets by `{prior_categorization_col}`, that will not form clusters together. (e.g., ATAC-seq: cluster neuron motifs only, then heart motifs only, then liver only)
- **Cluster on**: Apply a prior "clustering" that has already been done for us, defined in `{prior_categorization_col}`. (e.g., TF-MoDISco subpatterns)

*[Note: It is possible to apply **both** 'cluster within' and 'cluster on'. In this case, we ensure 'cluster within' is satisfied first, then 'cluster on' is applied next (e.g., ATAC-seq subpatterns: Ensure neuron subpatterns are kept separate. Then apply parent pattern clustering)]*

> CLI instructions:
```
python pipeline.py \
    --cluster-on {Metadata column to pre-cluster, then cluster on; str} \
    --cluster-within {Metadata column to slice motifs, then cluster; str} \
    ...
```

### 8-4-2. Motif clustering

We cluster motifs into **clusters**. (Refer to Tutorial #3: Motif clustering, for more detail)

Some suggested parameters for clustering:
- **Clustering algorithm**: Leiden algorithm, with Constant Potts quality metric `cpm_leiden` *(default)*
- **Similarity threshold**: `0.88` *(default)*
- **Recursive**: `True` *(default: `False`)*
- **Forced**: `True` *(default: `False`)*

> CLI instructions:
```
python pipeline.py \
    --sim-threshold {Similarity threshold to apply during main clustering; float} \
    --sim-threshold-force {Maximum similarity ensure between clusters, meta-clusters, sub-clusters; float} \
    --cluster-recursive {Apply recursive clusterig, until convergence; bool} \
    ...
```

### 8-4-3. (Optional) Meta-clustering

We cluster clusters into **meta-clusters**. Some suggested parameters for meta-clustering:
- **Clustering algorithm**: Leiden algorithm, with Constant Potts quality metric `cpm_leiden` *(default)*
- **Similarity threshold**: `0.88` *(default)*
- **Recursive**: `True` *(default: `False`)*
- **Forced**: `True` *(default: `False`)*

> CLI instructions:
```
python pipeline.py \
    --sim-threshold-meta {Similarity threshold to apply during meta-clustering; float} \
    ...
```

### 8-4-4. (Optional) Sub-clustering

We cluster motifs (within a cluster) into **sub-clusters**. Some suggested parameters for sub-clustering:
- **Clustering algorithm**: Leiden algorithm, with Constant Potts quality metric `cpm_leiden` *(default)*
- **Similarity threshold**: `0.88` *(default)*
- **Recursive**: `True` *(default: `False`)*
- **Forced**: `True` *(default: `False`)*

> CLI instructions:
```
python pipeline.py \
    --sim-threshold-sub {Similarity threshold to apply during sub-clustering; float} \
    ...
```

### 8-4-5. (Optional) Quality check

We can check the **quality of clustering**, by generating metrics, plots, and visualizations related to clustering quality. (Refer to **Tutorial #6 Additional analyses**, for more detail).

Ideally, we want the following: 
- **High similarity score within a cluster** (internal), to ensure the cluster is uniform 
- **Low similarity score between clusters** (external), to ensure each cluster is unique

When using quality, we generate the following metrics and plots, per clustering:
- **Comparison metrics and logos**: Record metrics and visualize the following motif logos, corresponding to: (1) Minimum internal similarity score, within a cluster, (2) Maximum external similarity score, between clusters:
    - (1) Minimum internal similarity: Lowest internal similarity score, Lowest internal similarity motif pairs (paired; motif A-B)
    - (2) Maximum external similarity: Highest external similarity score, Highest external similarity motif, Highest external similarity cluster
- **Histogram**: Plots two histogram: (1) Minimum internal similarity  score, within a cluster, (2) Maximum external similarity score, between clusters
- **Heatmap**: Plots a heatmap of maximum similarity score between all N x N pairs of motifs

> CLI instructions:
```
python pipeline.py \
    --quality {Generate quality metrics and plots to check clustering quality; bool}
    ...
```

### 8-4-6. Output:
This adds the following columns to the MotifCompendium metadata:
- **Motif clustering**: Clustering membership, per clustering process:
    - Cluster col: `{cluster_algorithm}_{sim_threshold}`
    - If `cluster_within` specified: `Within-{cluster_within}` + Cluster col
    - If `cluster_on` specified: `On-{cluster_on}` + Cluster col
    - If `recursive` specified: Cluster col + `-rec`
    - If `force` specified: Cluster col + `-force{sim_threshold_force}`
- **Meta-clustering**: Clustering membership, per clustering process:
    - Meta-cluster col: `{cluster_col}-meta{cluster_algoritm}_{sim_threshold}`
    - If `recursive` specified: Cluster col + `-rec`
    - If `force` specified: Cluster col + `-force{sim_threshold_force}`
- **Sub-clustering**: Clustering membership, per clustering process:
    - Sub-cluster col: `{cluster_col}-sub{cluster_algorithm}_{sim_threshold}`
    - If `recursive` specified: Cluster col + `-rec`
    - If `force` specified: Cluster col + `-force{sim_threshold_force}`

Outputs are saved at: 
- **Clustered** : `${OUTPUT_DIR}/motifcompendium_clustered.mc` - with the new clustering membership metadata columns
- (Optional) **Quality**: `${OUTPUT_DIR}/quality`
    - **Histogram**: `${OUTPUT_DIR}/quality/histogram_{cluster_col}`
    - **Heatmap**: `${OUTPUT_DIR}/quality/heatmap_{cluster_col}`

## 8-5. Average clusters

Fifth, we **average** each cluster. Clustering provides a membership for each motif. However, we often want a single representation of the cluster membership. Therefore, we take the (weighted) average of the membership to represent the cluster.

We average all three types of clusters, if present:
1. **Cluster**: From motif clustering (motifs --> clusters)
2. (Optional) **Meta-cluster**: From meta-clustering (clusters --> meta-clusters)
3. (Optional) **Sub-cluster**: From sub-clustering (motifs within a cluster --> sub-clusters)

During averaging, we aggregate **metadata** too. We do this by one of 5 types: (1) average, (2) sum, (3) count all, (4) count unique, (5) concatenate unique. These are pre-defined as follows:

| (Individual) Metadata column | Aggregation | (Aggregated) Metadata column |
|-----------------------------|-------------|-------------------------------|
| `name`                        | count       | `num_motifs`                    |
| `num_seqlets`                | sum         | `num_seqlets`                   |
| `model`                       | concat      | `model`                         |
| `avg_dist_summit`            | average     | `avg_dist_summit`              |
| `avg_contrib`                | average     | `avg_contrib`                  |
| `target`                      | concat      | `target`                        |
| `family`                      | concat      | `family`                        |
| `tissue`                      | concat      | `tissue`                        |
| `organ`                       | concat      | `organ`                         |
| `biosample`                   | concat      | `biosample`                     |
| `motifs`                     | concat      | `motifs`                        |
| `source`                      | concat      | `source`                        |



### 8-5-1. (Optional) Weighted average clusters

During averaging, we take the **weighted mean** of all the cluster's constituent motifs, by its subsequent constituent number of seqlets if the information exists:
- If `num_seqlets` *does* exists: Take the weighted mean, weighted by `num_seqlets`
- If `num_seqlets` does *not* exist: Take the standard,  unweighted mean

> CLI instructions: None

*Averaging is run automatically, as default*

### 8-5-2. Output:

This constructs a **new MotifCompendium object, per averaging**. Outputs are saved at:
- Cluster MotifCompendium: `${OUTPUT_DIR}/motifcompendium_avg.mc`
- Meta-cluster MotifCompendium: `${OUTPUT_DIR}/motifcompendium_metaavg.mc`
- Sub-cluster MotifCompendium: `${OUTPUT_DIR}/motifcompendium_subavg.mc`

## 8-6. Annotate clusters

Sixth, we **annotate clusters**, as we previously annotated motifs.

*(Same process as **8-2. Annotate motifs**; Refer to **8-2. Annotate motifs**)*

### 8-6-1. Output:

Outputs are saved at: 
- **Labeled Cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_avg_labeled.mc`
- (Optional) **Labeled Meta-cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_metaavg_labeled.mc`
- (Optional) **Labeled Sub-cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_subavg_labeled.mc`

## 8-7. Filter clusters

Seventh, we **filter clusters**, as we previously filtered motifs.

*(Same process as **8-3. Filter motifs**; Refer to **8-3. Filter motifs**)*

### 8-7-1. Output:

Outputs are saved at: 
- **Filtered Cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_avg_filtered.mc`
- (Optional) **Filtered Meta-cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_metaavg_filtered.mc`
- (Optional) **Filtered Sub-cluster MotifCompendium**: `${OUTPUT_DIR}/motifcompendium_subavg_filtered.mc`

## 8-8. Visualize motifs and cluster

Lastly, we **visualize** the motifs and clusters in HTML. (Refer to **Tutorial #4: Visualization**, for more detail)

We have two types of visualization reports:
1. **Summary table**: A tablular visualization of all motifs/clusters/meta-clusters/sub-clusters, that summarizes the logo, metadata, and images per row
2. **Collection list**: A list visualization of a cluster, showing all its constituent motifs

### 8-8-1. Summary table

This creates an **HTML file with a tabular format**. 

We can visualize all four types of motifs and clusters, if present:

1. **Motifs**: Individual motifs
2. **Cluster**: From motif clustering (motifs --> clusters)
3. (Optional) **Meta-cluster**: From meta-clustering (clusters --> meta-clusters)
4. (Optional) **Sub-cluster**: From sub-clustering (motifs within a cluster --> sub-clusters)

The following columns will be created, if present:
- **Logo**:
    - Forward, reverse complement motif
- **Metadata**: 
    - Default: `name`, `posneg`, `num_motifs`, `num_seqlets`, `avg_dist_summit`, `avg_contrib`, 
    - (Optional) Additional: `target`, `tissue`, `organ`, `system`, `source`, `motifs`
- **Annotation**:
    - `reference_name{i}`, `reference_score{i}`, `reference_logo{i}` for default: `i=2`
- (Optional) **Quality**:
    - `best_match_similarity`, `best_match_cluster`, `highest_external_similarity`,`highest_external_similarity_motif`, `highest_external_similarity_cluster`, `lowest_internal_similarity`, `lowest_internal_similarity_motif1`, `lowest_internal_similarity_motif2`

> CLI instructions:
```
python pipeline.py \
    --html-motif-table {Generate HTML summary table of filter-passed individual motifs; bool} \
    --html-motif-removed {Generate HTML summary table of filter-removed individual motifs; bool} \
    --html-cluster-table {Generate HTML summary table of filter-passed clusters, meta-clusters, sub-clusters; bool} \
    --html-cluster-removed {Generate HTML summary table of filter-removed clusters, meta-clusters, sub-clusters; bool} \
    ...
```

### 8-8-2. (Optional) Summary table: Max rows

As we deal with many motifs and many clusters, the HTML file can get very large. To prevent this, we can limit the maximum number of rows for all Summary Table:
- **HTML max rows**: Maximum number of rows to generate in HTML Summary Table

> CLI instructions:
```
python pipeline.py \
    --html-max-rows {Maximum number of rows to generate in HTML Summary Table; int} \
    ...
```

### 8-8-3. Collection list

This creates a **list of logos** of the constituent motifs of each cluster, into two long columns.

*(Note: As we deal with many motifs and many clusters, the HTML file can get very large)*

> CLI instructions:
```
python pipeline.py \
    --html-motif-collection {Generate HTML collection list of all constituents motifs for each cluster; bool} \
    ...
```

### 8-8-4. Output:

This constructs an HTML file, per specified visualization. Outputs are saved at: `${OUTPUT_DIR}/html`
- **Summary table**:
    - **Motif** (Passed): `${OUTPUT_DIR}/html/motifcompendium_motif_table.html`
    - **Motif** (Removed): `${OUTPUT_DIR}/html/motifcompendium_motif_removed_table.html`
    - **Cluster** (Passed): `${OUTPUT_DIR}/html/motifcompendium_avg_table.html`
    - **Cluster** (Removed): `${OUTPUT_DIR}/html/motifcompendium_avg_removed_table.html`
    - **Meta-cluster** (Passed): `${OUTPUT_DIR}/html/motifcompendium_metaavg_table.html`
    - **Meta-cluster** (Removed): `${OUTPUT_DIR}/html/motifcompendium_metaavg_removed_table.html`
    - **Sub-cluster** (Passed): `${OUTPUT_DIR}/html/motifcompendium_subavg_table.html`
    - **Sub-cluster** (Removed): `${OUTPUT_DIR}/html/motifcompendium_subavg_removed_table.html`
- **Collection list**: `${OUTPUT_DIR}/html/motifcompendium_motif_collection.html`

## 8-9. (Optional) Additional pipeline configurations

Many parameters and configurations were **fixed** in this tutorial. Additionally, there are many other configurations that can be needed in the pipeline. 

- For advanced users, we have enabled **direct access** to many parameters and configurations, beyond the basic CLI interface at: `./pipeline/configs.py` - feel free to modify as needed.
- Note: `./pipeline/configs.py` is directly imported during each run of `./pipeline/pipeline.py`, so modify, then save before each run.

Lastly, MotifCompendium is designed to be a highly flexible package. For further modifications, please feel free to **develop your own pipeline**, referring to `./pipeline/pipeline.py` as a guideline.